# 🧩 13 · Averaging and classification: reference-free iteration and maximum likelihood
### How thousands of noisy copies become one clear structure, and how to sort copies that are not all the same

**BiologyToResearchV3** · Part C — Classical computational cryo-ET (2009–2019) · Module 13 of 26

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/cryoem/tutorial/BiologyToResearchV3/notebooks/13_averaging_classification_ml.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier, CPU only (about 4–6 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, SciPy, Matplotlib, scikit-learn |
| **Prerequisites** | Module 04 (mixture models, EM, clustering, silhouette), Module 11 (constrained correlation, fast rotational search). Helpful: 07 (missing wedge), 09 (FSC) |
| **Estimated time** | 150 min (reading + running + the ✏️ exercises) |
| **Lab papers covered** | Xu, Beck & Alber 2012 (alignment-through-classification); Zhao, Zeng, Guo & Xu 2018 (FAML); Kim *et al.* 2023 (review: unbiased pattern mining); Thalassinos *et al.* 2013 (perspective: conformational states). See [`PAPER_ATLAS.md`](../PAPER_ATLAS.md) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> All experiments are small, controlled *toy versions* of published methods. They illustrate the core idea of each
> paper; they do **not** reproduce the papers' numbers. To keep everything fast on a CPU, Sections 3–6 use a **2D
> analogue** ("flatland") of subtomograms; this is said clearly wherever it happens. Molecular densities come from real
> PDB entries via the open-source PolNet simulator (offline: simple stand-in shapes, and the notebook says so).

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Explain** why averaging $N$ aligned copies raises the signal-to-noise ratio about $N$-fold, and **implement**
   missing-wedge-weighted averaging in Fourier space.
2. **Demonstrate** reference bias ("Einstein from noise") and explain why the field moved toward template-free methods.
3. **Implement** a 2D version of the *alignment-through-classification* loop of Xu, Beck & Alber (2012): pairwise
   constrained correlation → hierarchical clustering cut at the best silhouette → class averages → re-align → re-assign,
   with feature-enhancement weights.
4. **Write down** the generative model $X_i = R_{\phi_i}A_{\kappa_i} + G_i$ of maximum-likelihood classification and
   explain its E-step and M-step in words.
5. **Implement** a FAML-style EM (Zhao *et al.* 2018) whose pose integral uses a small set of candidate poses from fast
   rotational search, weighted by Monte-Carlo Voronoi volumes, and **compare** it with hard best-pose assignment as the
   SNR drops.
6. **Measure** the occupancy of two compositional states (20S vs capped proteasome) with an uncertainty, and locate
   the difference between the states with a difference map.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | check the computer, load tools and molecules | 5 min |
| 1 · Averaging | average 3D subtomograms; fill the missing wedge by Fourier weighting | 20 min |
| 2 · Reference bias | make a ribosome appear out of pure noise | 15 min |
| 3 · Flatland | a fast 2D stand-in for subtomograms, with a 2D fast aligner | 15 min |
| 4 · Alignment through classification | Xu, Beck & Alber 2012, step by step | 30 min |
| 5 · Maximum likelihood (FAML) | soft EM over candidate poses vs hard assignment | 35 min |
| 6 · From classes to biology | how many proteasomes carry a cap? | 15 min |
| ✅ / 🧠 / 🚀 | checkpoint, discussion, going further | 15 min |

## 🧫 The biological question

The 20S proteasome is the barrel in which cells shred unwanted proteins. In a cell, some barrels are bare, some carry a
regulatory **cap** on one end, some on both ends; ribosomes cycle through states as they translate; chaperones open and
close. A single cryo-ET subtomogram is far too noisy to tell these apart: its signal-to-noise ratio is around 0.01–0.1.
The only way forward is to **average many copies**, but averaging copies that are not the same produces a blurred
chimera. So we must **sort while we align** (and align while we sort). How do you sort thousands of noisy, randomly
oriented, partly missing 3D images into groups when you do not know what the groups look like, or even how many there
are? That is the problem of subtomogram **classification**, and this notebook builds the two classical answers from the
Xu Lab: iterative alignment-through-classification (2012) and maximum likelihood with fast alignment (FAML, 2018).

---
## 0 · Environment setup

Run the cells in order, from the top (**Shift + Enter**). The first cell reports what computer you are on.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected (fine: this notebook only needs a CPU).")
try:
    import torch
    print("PyTorch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
except ImportError:
    print("PyTorch not installed (not needed in this notebook).")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("matplotlib", "matplotlib"),
                         ("sklearn", "scikit-learn")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import os
import time

import matplotlib.pyplot as plt
import numpy as np
from scipy import ndimage, sparse
from scipy.cluster.hierarchy import dendrogram, fcluster, linkage
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import squareform
from scipy.special import logsumexp
from sklearn.metrics import adjusted_rand_score, silhouette_score

SEED = 0                                   # one seed → identical results on every run
rng = np.random.default_rng(SEED)
DEVICE = "cpu"                             # no deep learning here; kept for course-wide consistency
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "image.cmap": "gray"})
print("NumPy", np.__version__, "| seed", SEED, "| device", DEVICE)

### 🧰 Course toolkit

The helpers below are copied verbatim from [`course_toolkit.py`](../course_toolkit.py) so that this notebook runs on
its own in Colab: downloading the molecular zoo, rotating volumes and images, Fourier masks (missing wedge), noise,
correlation, FSC and clustering accuracy. Just run the cell.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v1.0; copied so this notebook runs alone) ──
import struct
import urllib.request

# Density templates made from PDB entries at 10 Å/voxel, distributed with the
# PolNet cryo-ET simulator (Martinez-Sanchez et al., Apache-2.0 licence).
# The commit is pinned so the files never change underneath the course.
POLNET_COMMIT = "73e613cc9162d26c7fd740ce3299bcb280a3e3b6"


POLNET_URL = ("https://raw.githubusercontent.com/anmartinezs/polnet/"
              + POLNET_COMMIT + "/data/templates/{kind}/{name}.{ext}")


CACHE_DIR = os.environ.get("CRYO_COURSE_CACHE", "course_data")


# The "molecular zoo": names/organisms/masses from PolNet docs/molecules_table.md
ZOO = {
    "4v4r": dict(name="70S ribosome", organism="bacterial (E. coli / T. thermophilus)", mass_kDa=2277, symmetry="C1"),
    "4v7r": dict(name="80S ribosome", organism="S. cerevisiae", mass_kDa=5851, symmetry="C1"),
    "5mrc": dict(name="mitochondrial ribosome", organism="S. cerevisiae", mass_kDa=3325, symmetry="C1"),
    "3j9i": dict(name="20S proteasome", organism="Thermoplasma acidophilum", mass_kDa=659, symmetry="D7"),
    "6utj": dict(name="singly capped proteasome", organism="Thermoplasma acidophilum", mass_kDa=839, symmetry="C7"),
    "4cr2": dict(name="26S proteasome", organism="S. cerevisiae", mass_kDa=1309, symmetry="C1"),
    # NB: PolNet's 4V94 map holds TWO TRiC complexes side by side (1.95 MDa ≈ 2 × ~1 MDa).
    "4v94": dict(name="TRiC/CCT chaperonin (×2)", organism="S. cerevisiae", mass_kDa=1952, symmetry="C2"),
    "1bxn": dict(name="RuBisCO", organism="Cupriavidus necator", mass_kDa=280, symmetry="D4"),
    "1qvr": dict(name="ClpB chaperone", organism="Thermus thermophilus", mass_kDa=297, symmetry="C1"),
    "3cf3": dict(name="p97/VCP ATPase", organism="Mus musculus", mass_kDa=None, symmetry="C6"),
    "2cg9": dict(name="Hsp90–Sba1 chaperone", organism="S. cerevisiae", mass_kDa=None, symmetry="C2"),
    "1u6g": dict(name="Cand1–Cul1–Roc1 ligase", organism="H. sapiens", mass_kDa=239, symmetry="C1"),
    "3d2f": dict(name="Sse1p–Hsp70 chaperone", organism="S. cerevisiae / H. sapiens", mass_kDa=236, symmetry="C1"),
    "1s3x": dict(name="Hsp70 ATPase domain", organism="H. sapiens", mass_kDa=43, symmetry="C1"),
}


def _download(url: str, dest: str, timeout: float = 60.0) -> bool:
    """Download ``url`` to ``dest`` (no-op if it exists). Returns True on success."""
    if os.path.exists(dest) and os.path.getsize(dest) > 0:
        return True
    os.makedirs(os.path.dirname(dest) or ".", exist_ok=True)
    try:
        with urllib.request.urlopen(url, timeout=timeout) as r, open(dest + ".part", "wb") as f:
            f.write(r.read())
        os.replace(dest + ".part", dest)
        return True
    except Exception as exc:  # network blocked, offline, ...
        print(f"  ⚠️  download failed for {url.split('/')[-1]}: {type(exc).__name__}")
        return False


def read_mrc(path_or_bytes) -> tuple[np.ndarray, float]:
    """Minimal MRC2014 reader (modes 0, 1, 2, 6). Returns (array[z, y, x], voxel_size_Å).

    The MRC header is 1024 bytes of little-endian 32-bit words; words 1–3 are the
    grid size (nx, ny, nz), word 4 the data type, words 11–13 the cell size in Å,
    and word 24 the length of an optional extended header.
    """
    raw = path_or_bytes if isinstance(path_or_bytes, (bytes, bytearray)) else open(path_or_bytes, "rb").read()
    if raw[:2] == b"\x1f\x8b":  # gzip
        import gzip
        raw = gzip.decompress(raw)
    nx, ny, nz, mode = struct.unpack("<4i", raw[0:16])
    mx, my, mz = struct.unpack("<3i", raw[28:40])
    cella = struct.unpack("<3f", raw[40:52])
    mapc, mapr, maps = struct.unpack("<3i", raw[64:76])
    nsymbt = struct.unpack("<i", raw[92:96])[0]
    dtype = {0: np.int8, 1: np.int16, 2: np.float32, 6: np.uint16}[mode]
    data = np.frombuffer(raw, dtype=dtype, offset=1024 + nsymbt, count=nx * ny * nz)
    vol = data.reshape(nz, ny, nx).astype(np.float32)
    if (mapc, mapr, maps) != (1, 2, 3) and (mapc, mapr, maps) != (0, 0, 0):
        # reorder axes so that the array is [z, y, x]
        order = [3 - maps, 3 - mapr, 3 - mapc]
        vol = np.transpose(vol, np.argsort(order))
    voxel = float(cella[0] / mx) if mx > 0 else 1.0
    return vol, voxel


def fetch_zoo_map(pdb_id: str, voxel: str = "10A") -> tuple[np.ndarray, float, str]:
    """Return (density[z, y, x], voxel_size_Å, source) for a molecular-zoo entry.

    Tries the pinned PolNet template first; if the download fails (no internet),
    falls back to a *procedural* stand-in so the notebook still runs.
    """
    pdb_id = pdb_id.lower()
    dest = os.path.join(CACHE_DIR, f"polnet_{voxel}_{pdb_id}.mrc")
    url = POLNET_URL.format(kind=f"mrcs_{voxel}", name=pdb_id, ext="mrc")
    if _download(url, dest):
        vol, vs = read_mrc(dest)
        return vol, vs, f"PolNet template from PDB {pdb_id.upper()}"
    vol = procedural_molecule(pdb_id)
    return vol, 10.0, f"procedural stand-in for {pdb_id.upper()} (offline fallback)"


def procedural_molecule(pdb_id: str, size: int = 40) -> np.ndarray:
    """Offline fallback: a coarse shape with the right size/symmetry flavour (1 nm voxels)."""
    g = (np.arange(size) - size / 2 + 0.5)
    z, y, x = np.meshgrid(g, g, g, indexing="ij")
    r = np.sqrt(x**2 + y**2)

    def blob(cz, cy, cx, s):
        return np.exp(-((z - cz)**2 + (y - cy)**2 + (x - cx)**2) / (2 * s**2))
    pid = pdb_id.lower()
    if pid in ("4v4r", "4v7r", "5mrc"):          # two unequal subunits
        v = blob(-3, 0, 0, 7.5) + 0.8 * blob(6, 2, 1, 5.5)
    elif pid in ("3j9i", "6utj", "4cr2"):        # stacked rings / barrel
        v = np.exp(-((r - 5.5)**2) / 4.0) * (np.abs(z) < 7.5)
        if pid != "3j9i":
            v = v + 0.8 * blob(10, 0, 0, 4)
    elif pid == "4v94":                          # double ring with cavity
        v = np.exp(-((r - 7)**2) / 6.0) * (np.abs(z) < 8)
    elif pid in ("3cf3", "1qvr"):                # hexameric ring
        ang = np.arctan2(y, x)
        v = np.exp(-((r - 5)**2) / 3.0) * (np.abs(z) < 4) * (1 + 0.5 * np.cos(6 * ang))
    elif pid == "1bxn":                          # compact ball
        v = blob(0, 0, 0, 4.5)
    else:                                        # small asymmetric blob
        v = blob(0, 0, 0, 3) + 0.5 * blob(2, 2, 0, 2)
    return ndimage.gaussian_filter(v.astype(np.float32), 1.0)


def prepare_map(vol: np.ndarray, size: int = 32, sigma: float = 1.0) -> np.ndarray:
    """Low-pass, centre (by centre of mass), crop/pad to ``size``³ and scale to [0, 1].

    PolNet templates are sharp sums of atoms; a Gaussian blur of ~1 voxel (1 nm)
    mimics the resolution of a cellular tomogram.
    """
    v = ndimage.gaussian_filter(np.clip(vol, 0, None).astype(np.float32), sigma)
    com = np.array(ndimage.center_of_mass(v))
    shift = np.array(v.shape) / 2 - 0.5 - com
    v = ndimage.shift(v, shift, order=1)
    out = np.zeros((size, size, size), np.float32)
    s = np.array(v.shape)
    src0 = np.maximum((s - size) // 2, 0)
    dst0 = np.maximum((size - s) // 2, 0)
    n = np.minimum(s, size)
    out[dst0[0]:dst0[0] + n[0], dst0[1]:dst0[1] + n[1], dst0[2]:dst0[2] + n[2]] = \
        v[src0[0]:src0[0] + n[0], src0[1]:src0[1] + n[1], src0[2]:src0[2] + n[2]]
    out -= out.min()
    return out / (out.max() + 1e-12)


def load_zoo(ids=("4v4r", "3j9i", "6utj", "1bxn"), size: int = 32, sigma: float = 1.0,
             verbose: bool = True) -> dict:
    """Fetch and prepare several zoo maps. Returns {pdb_id: volume(size³)}."""
    zoo = {}
    for pid in ids:
        raw, vs, src = fetch_zoo_map(pid)
        zoo[pid] = prepare_map(raw, size=size, sigma=sigma)
        if verbose:
            print(f"  {pid.upper():5s} {ZOO.get(pid, {}).get('name', ''):28s} raw {raw.shape} @ {vs:.0f} Å/voxel  ← {src}")
    return zoo


def rot_z(a):
    c, s = np.cos(a), np.sin(a)
    return np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])


def random_rotation(rng: np.random.Generator) -> np.ndarray:
    """Uniformly random proper rotation (det = +1) from a random unit quaternion."""
    q = rng.normal(size=4)
    q /= np.linalg.norm(q)
    w, x, y, z = q
    return np.array([
        [1 - 2 * (y * y + z * z), 2 * (x * y - z * w), 2 * (x * z + y * w)],
        [2 * (x * y + z * w), 1 - 2 * (x * x + z * z), 2 * (y * z - x * w)],
        [2 * (x * z - y * w), 2 * (y * z + x * w), 1 - 2 * (x * x + y * y)]])


_P = np.array([[0, 0, 1], [0, 1, 0], [1, 0, 0]], float)  # (x,y,z) <-> (z,y,x)


def rotate_volume(vol: np.ndarray, R: np.ndarray, shift_xyz=(0, 0, 0), order: int = 1,
                  cval: float = 0.0) -> np.ndarray:
    """Rotate a volume about its centre by R (then translate by shift, in voxels, (x,y,z)).

    out(p) = vol(Rᵀ (p − c − t) + c): the *object* is rotated by R and moved by t.
    """
    R_idx = _P @ R @ _P                       # same rotation written in (z, y, x) index order
    c = (np.array(vol.shape) - 1) / 2.0
    t_idx = _P @ np.asarray(shift_xyz, float)
    M = R_idx.T                               # inverse rotation maps output -> input
    offset = c - M @ (c + t_idx)
    return ndimage.affine_transform(vol, M, offset=offset, order=order, mode="constant", cval=cval)


def rotate_image(img: np.ndarray, angle_rad: float, shift_xy=(0, 0), order: int = 1) -> np.ndarray:
    """2D analogue of :func:`rotate_volume` (counter-clockwise in the (x, y) plane)."""
    c, s = np.cos(angle_rad), np.sin(angle_rad)
    R = np.array([[c, -s], [s, c]])
    R_idx = R[::-1, ::-1]                      # (x,y) -> (y,x)
    ctr = (np.array(img.shape) - 1) / 2.0
    t_idx = np.asarray(shift_xy, float)[::-1]
    M = R_idx.T
    return ndimage.affine_transform(img, M, offset=ctr - M @ (ctr + t_idx), order=order, mode="constant")


def freq_grid(shape):
    """Frequency coordinates (cycles/voxel) for an unshifted FFT; returns list per axis."""
    return np.meshgrid(*[np.fft.fftfreq(n) for n in shape], indexing="ij")


def radial_frequency(shape) -> np.ndarray:
    """|k| in cycles/voxel for every coefficient of an unshifted FFT."""
    return np.sqrt(sum(k**2 for k in freq_grid(shape)))


def missing_wedge_mask(shape, tilt_max_deg: float = 60.0) -> np.ndarray:
    """Binary Fourier mask of a single-axis tilt series about y (±tilt_max).

    A coefficient (k_z, k_y, k_x) is measured if |k_z| ≤ |k_x|·tan(tilt_max)
    — the formula M(ξ) = 1(|ξ_3| ≤ |ξ_1| tan θ) used by Xu et al. (2012).
    For 2D images [z, x] pass a 2-tuple shape.
    """
    k = freq_grid(shape)
    kz, kx = k[0], k[-1]
    m = np.abs(kz) <= np.abs(kx) * np.tan(np.radians(tilt_max_deg)) + 1e-9
    m[(0,) * len(shape)] = True               # keep the mean (DC) term
    return m.astype(np.float32)


def apply_fourier_mask(vol: np.ndarray, mask: np.ndarray) -> np.ndarray:
    """Multiply the FFT of ``vol`` by ``mask`` (unshifted layout) and return the real part."""
    return np.real(np.fft.ifftn(np.fft.fftn(vol) * mask)).astype(np.float32)


def lowpass(vol: np.ndarray, cutoff: float, soft: float = 0.02) -> np.ndarray:
    """Soft low-pass filter; ``cutoff`` in cycles/voxel (0.5 = Nyquist)."""
    k = radial_frequency(vol.shape)
    filt = 1 / (1 + np.exp((k - cutoff) / soft))
    return apply_fourier_mask(vol, filt)


def add_noise_snr(signal: np.ndarray, snr: float, rng: np.random.Generator) -> np.ndarray:
    """Add white Gaussian noise so that var(signal)/var(noise) = snr."""
    sigma = np.sqrt(signal.var() / snr)
    return (signal + rng.normal(0, sigma, signal.shape)).astype(np.float32)


def normalize(v: np.ndarray) -> np.ndarray:
    """Zero mean, unit standard deviation."""
    return ((v - v.mean()) / (v.std() + 1e-12)).astype(np.float32)


def pearson(a: np.ndarray, b: np.ndarray) -> float:
    """Pearson correlation of two arrays (flattened)."""
    a = a.ravel() - a.mean()
    b = b.ravel() - b.mean()
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))


def fsc(v1: np.ndarray, v2: np.ndarray, pixel_A: float = 10.0, n_shells: int | None = None):
    """Fourier shell (ring, in 2D) correlation. Returns (frequency_1/Å, fsc, n_coeffs)."""
    F1, F2 = np.fft.fftn(v1), np.fft.fftn(v2)
    k = radial_frequency(v1.shape)
    n_shells = n_shells or min(v1.shape) // 2
    idx = np.minimum((k * 2 * n_shells).astype(int), n_shells)
    num = np.bincount(idx.ravel(), (F1 * np.conj(F2)).real.ravel(), n_shells + 1)
    d1 = np.bincount(idx.ravel(), (np.abs(F1)**2).ravel(), n_shells + 1)
    d2 = np.bincount(idx.ravel(), (np.abs(F2)**2).ravel(), n_shells + 1)
    cnt = np.bincount(idx.ravel(), minlength=n_shells + 1)
    curve = num / np.sqrt(d1 * d2 + 1e-20)
    freq = (np.arange(n_shells + 1) + 0.5) / (2 * n_shells) / pixel_A
    return freq[:n_shells], curve[:n_shells], cnt[:n_shells]


def resolution_at(freq, curve, threshold: float = 0.143) -> float:
    """First crossing of the FSC curve below ``threshold`` → resolution in Å."""
    below = np.where(curve[1:] < threshold)[0]
    if len(below) == 0:
        return float(1 / freq[-1])
    i = below[0] + 1
    f0, f1, c0, c1 = freq[i - 1], freq[i], curve[i - 1], curve[i]
    f = f0 + (threshold - c0) * (f1 - f0) / (c1 - c0 + 1e-12)
    return float(1 / f)


def hungarian_accuracy(y_true, y_pred) -> float:
    """Clustering accuracy after the best one-to-one relabelling (Hungarian algorithm)."""
    from scipy.optimize import linear_sum_assignment
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    ct = np.zeros((y_pred.max() + 1, y_true.max() + 1))
    np.add.at(ct, (y_pred, y_true), 1)
    r, c = linear_sum_assignment(-ct)
    return float(ct[r, c].sum() / len(y_true))


def show_orthoslices(vol, title: str = "", axes=None, cmap: str = "gray"):
    """Show the three central orthogonal slices (xy, xz, yz) of a volume."""
    import matplotlib.pyplot as plt
    if axes is None:
        _, axes = plt.subplots(1, 3, figsize=(7.5, 2.6))
    z, y, x = (np.array(vol.shape) // 2)
    for ax, im, lab in zip(axes, [vol[z], vol[:, y, :], vol[:, :, x]], ["xy (z mid)", "xz (y mid)", "yz (x mid)"]):
        ax.imshow(im, cmap=cmap)
        ax.set_title(f"{title} {lab}".strip(), fontsize=8)
        ax.axis("off")
    return axes


print("🧰 toolkit loaded")

### The molecules

We use four members of the course's molecular zoo: the bacterial **70S ribosome** (4V4R), the **20S proteasome**
(3J9I), the **singly capped proteasome** (6UTJ: the same barrel plus a regulatory cap, i.e. a *compositional variant*)
and **RuBisCO** (1BXN). Section 1 uses 3D maps at 32³ voxels of 1 nm; later sections use 2D projections of the same maps.

In [ ]:
# ── Load the cast (3D maps, 32³ voxels of 10 Å) ──
t0 = time.time()
CAST = ["4v4r", "3j9i", "6utj", "1bxn"]
zoo3d = load_zoo(CAST, size=32, sigma=1.0)
print(f"loaded {len(zoo3d)} maps in {time.time() - t0:.1f} s")
axes = plt.subplots(1, 3, figsize=(7.5, 2.6))[1]
show_orthoslices(zoo3d["4v4r"], "ribosome", axes=axes)
plt.suptitle("Our reference ribosome (4V4R), central slices; 1 voxel = 1 nm", fontsize=9)
plt.tight_layout(); plt.show()

---
## 1 · Averaging: why many noisy copies make one clear structure

> 📍 **Where we are.** Module 11 taught you to *align* two subtomograms quickly (constrained correlation, fast
> rotational search) and Module 12 to refine and scale alignment. Here we use alignments to **average** copies, and then
> to **sort** copies into classes. Module 14 will go one step further: discovering classes nobody knew existed.

### 1.1 The averaging law

Suppose subtomogram $i$ is the true structure $s$ plus independent noise $n_i$ with variance $\sigma^2$, all already
aligned. The average of $N$ copies is

$$\bar f = \frac1N\sum_{i=1}^N (s + n_i) = s + \frac1N\sum_{i=1}^N n_i, \qquad \operatorname{Var}\Big(\frac1N\sum_i n_i\Big)=\frac{\sigma^2}{N}.$$

*In words:* the signal is the same in every copy, so it survives averaging, while independent noise partly cancels; the
noise **variance** falls as $1/N$, so the SNR (a variance ratio, Module 04) **grows $N$-fold**. 1,000 copies at SNR 0.01
give an average at SNR ≈ 10. This is why subtomogram averaging can reach a few ångströms from tomograms that look like
snow.

### 1.2 The missing wedge does not average away, unless you weight it

Every subtomogram misses the same **wedge** of Fourier space *in the microscope frame* (Module 07): directions the stage
could not tilt to. After we rotate each copy back into a common frame, the wedges point in **different** directions, so
together they can cover all of Fourier space. If we simply average in real space, a Fourier coefficient seen by only 5
of 40 copies is divided by 40 and comes out 8× too weak. The fix (used throughout the Xu Lab's averaging code) is to
average **each Fourier coefficient over the copies that actually measured it**:

$$\hat A(\xi) \;=\; \frac{\sum_{i=1}^N M_i'(\xi)\,F_i'(\xi)}{\max\!\big(n(\xi),\,1\big)}, \qquad n(\xi)=\sum_{i=1}^N M_i'(\xi).$$

Here $F_i'$ is the Fourier transform of copy $i$ after rotating it back, $M_i'(\xi)\in\{0,1\}$ says whether copy $i$
measured frequency $\xi$ (its wedge mask, rotated along with it), and $n(\xi)$ counts the copies that measured $\xi$.
*In words:* for every frequency, divide by the number of witnesses, not by the number of copies.

🧮 **Which coefficients did a rotated copy measure?** If copy $i$ was made by rotating the structure by $R_i$ and then
losing the wedge $M$ (in the microscope frame), then after rotating it back by $R_i^{\mathsf T}$ its coefficient at
frequency $\xi$ is the original coefficient at $R_i\xi$. So $M_i'(\xi) = M(R_i\xi)$: we rotate the frequency vector and
ask whether it lands inside the measured region $|\xi_z| \le |\xi_x|\tan 60°$.

In [ ]:
# ── 1.2 The wedge mask of a copy after rotating it back ──
def rotated_wedge_mask_3d(shape, R, tilt_max_deg=60.0):
    """M'(ξ) = M(R ξ): which Fourier coefficients a copy made with rotation R measured, seen in the common frame."""
    kz, ky, kx = freq_grid(shape)
    k = np.stack([kx, ky, kz])                       # physical (x, y, z) order, as R expects
    kr = np.tensordot(R, k, axes=1)                  # R ξ for every frequency at once
    m = np.abs(kr[2]) <= np.abs(kr[0]) * np.tan(np.radians(tilt_max_deg)) + 1e-9
    m[0, 0, 0] = True                                # the mean (DC) is always measured
    return m.astype(np.float32)


def make_subtomogram(template, R, snr, rng, tilt_max_deg=60.0):
    """Rotate, cut the missing wedge, add noise (snr=None: no noise); noise is confined to the wedge, as in a reconstruction."""
    M = missing_wedge_mask(template.shape, tilt_max_deg)
    clean = apply_fourier_mask(rotate_volume(template, R), M)
    return clean if snr is None else apply_fourier_mask(add_noise_snr(clean, snr, rng), M)


# Sanity check with white noise: after rotating back, the energy should sit inside M(Rξ), not inside M(Rᵀξ).
R_test = random_rotation(rng)
back = rotate_volume(apply_fourier_mask(rng.normal(size=(32, 32, 32)), missing_wedge_mask((32, 32, 32))), R_test.T)
power = np.abs(np.fft.fftn(back))**2
right = 1 - (power * rotated_wedge_mask_3d(back.shape, R_test)).sum() / power.sum()
wrong = 1 - (power * rotated_wedge_mask_3d(back.shape, R_test.T)).sum() / power.sum()
print(f"energy outside the predicted mask: {100 * right:.1f} % (correct formula)  vs  {100 * wrong:.1f} % (wrong formula)")
assert right < 0.5 * wrong, "the rotated-wedge formula should explain the data much better than its transpose"

🔍 **What just happened?** We rotated a wedge-filtered noise volume and checked which Fourier coefficients are non-zero
afterwards. The formula $M(R\xi)$ leaves only about a tenth of the energy outside (linear interpolation during the
rotation smears a little), while the tempting-but-wrong $M(R^{\mathsf T}\xi)$ leaves about three times more. Small checks like this catch the
most common bug in averaging code: rotating a mask the wrong way.

### 1.3 Experiment: naive vs wedge-weighted averaging

We now pretend alignment is solved (we use the true rotations, an "oracle"), so that we isolate the averaging step.
Three situations: few copies (3), many copies (40) in random orientations, and 40 copies with a **preferred
orientation**: all rotated only about the beam axis $z$, like membrane proteins lying flat in a membrane.

In [ ]:
# ── 1.3 Average oracle-aligned subtomograms, naive vs weighted ──
def average_subtomograms(template, rotations, snr, rng, tilt_max_deg=60.0):
    """Return (naive real-space average, wedge-weighted Fourier average, coverage count n(ξ)) of oracle-aligned copies."""
    acc = np.zeros(template.shape, complex)
    count = np.zeros(template.shape, np.float32)
    naive = np.zeros(template.shape, np.float32)
    for R in rotations:
        sub = make_subtomogram(template, R, snr, rng, tilt_max_deg)
        back = rotate_volume(sub, R.T)                # undo the (known) rotation
        Mi = rotated_wedge_mask_3d(template.shape, R, tilt_max_deg)
        naive += back / len(rotations)
        acc += np.fft.fftn(back) * Mi
        count += Mi
    weighted = np.real(np.fft.ifftn(np.where(count > 0, acc / np.maximum(count, 1), 0))).astype(np.float32)
    return naive, weighted, count


t0 = time.time()
T = zoo3d["4v4r"]
scenarios = {
    "3 copies, random": [random_rotation(rng) for _ in range(3)],
    "40 copies, random": [random_rotation(rng) for _ in range(40)],
    "40 copies, preferred (about z only)": [rot_z(rng.uniform(0, 2 * np.pi)) for _ in range(40)],
}
results = {}
print("correlation with the true map         no noise: naive / weighted     SNR 0.1: naive / weighted   never measured")
for name, rots in scenarios.items():
    clean_naive, clean_weighted, count = average_subtomograms(T, rots, snr=None, rng=rng)   # geometry only
    naive, weighted, _ = average_subtomograms(T, rots, snr=0.1, rng=rng)                     # realistic noise
    results[name] = (naive, weighted, count)
    print(f"{name:38s}   {pearson(clean_naive, T):.3f} / {pearson(clean_weighted, T):.3f}"
          f"              {pearson(naive, T):.3f} / {pearson(weighted, T):.3f}         {100 * (count == 0).mean():4.1f} %")
print(f"({time.time() - t0:.1f} s)")

In [ ]:
# ── 1.3 Figure: Fourier coverage and the resulting averages ──
fig, axes = plt.subplots(2, 3, figsize=(9, 5.6))
for j, (name, (naive, weighted, count)) in enumerate(results.items()):
    cov = np.fft.fftshift(count)[:, 16, :]                   # the (k_z, k_x) plane at k_y = 0
    im = axes[0, j].imshow(cov, cmap="viridis", vmin=0, vmax=count.max())
    axes[0, j].set_title(f"coverage n(ξ): {name}", fontsize=8)
    axes[0, j].set_xlabel("$k_x$"); axes[0, j].set_ylabel("$k_z$")
    plt.colorbar(im, ax=axes[0, j], fraction=0.046)
    axes[1, j].imshow(weighted[:, 16, :])
    axes[1, j].set_title(f"weighted average (SNR 0.1), xz slice\ncorr with truth {pearson(weighted, T):.2f}", fontsize=8)
    axes[1, j].set_xlabel("x (nm)"); axes[1, j].set_ylabel("z (nm)")
plt.suptitle("Missing-wedge-weighted averaging of oracle-aligned ribosome subtomograms (SNR 0.1, ±60° tilt)", fontsize=9)
plt.tight_layout(); plt.show()

🔍 **What just happened?**

- **Top row**: how many copies measured each Fourier coefficient (a slice through Fourier space; the centre is low
  frequency). With 3 copies the coverage is patchy; with 40 randomly oriented copies every direction is measured many
  times. With a **preferred orientation** a double cone around $k_z$ stays at **zero**: no copy ever measured it.
- **Printed numbers, no noise**: here only geometry matters, and weighting beats the naive average, most clearly with
  few copies (uneven coverage). With many random orientations the coverage is nearly uniform, so the naive average is
  almost a scaled copy of the weighted one and the two agree.
- **Printed numbers, SNR 0.1**: the advantage shrinks or even reverses. A coefficient measured by a single noisy copy is
  mostly noise, and weighting gives it full strength; the naive average quietly damps it. This is a bias–variance
  trade-off, and practical codes guard against it: AITom's SSNR code, for example, ignores Fourier coefficients seen by
  fewer than two subtomograms, and Wiener-like schemes divide by $n(\xi)+\lambda$ instead of $n(\xi)$.
- **Bottom right**: weighting cannot invent what was never measured. Particles stuck in one orientation leave a
  *missing cone* and the average stays smeared along $z$. This is a real problem for membrane proteins and for flat
  specimens; it is one reason why the *distribution of orientations* is always reported in subtomogram-averaging papers.

✏️ **Your turn.** Change the tilt range by adding `tilt_max_deg=40.0` to the `average_subtomograms(...)` call in 1.3
(and re-run the figure cell). How does the never-measured fraction change for the preferred-orientation case?

---
## 2 · Reference bias: "Einstein from noise"

Averaging needs alignment, and alignment needs something to align **to**: a reference. The obvious choice is a known
structure (a template). Here is the trap. Take images that contain **no molecule at all**, only noise. For each one,
search all rotations and shifts for the pose where it correlates best with the template; then average the images in
those poses. Because we *chose* each pose to maximise agreement with the template, each image contributes the noise
pattern that happens to look most like the template. Average enough of them and the template appears.

🧮 A one-line way to see it: if $\phi_i^\ast=\arg\max_\phi\langle T_\phi n_i,\,t\rangle$ (the pose that best matches
template $t$), then the aligned noise $T_{\phi_i^\ast}n_i$ is no longer zero-mean: its average points toward $t$. In
words: **selection creates signal**. The famous version of this experiment produced a portrait of Einstein from pure
noise, hence the name. (It is a cautionary tale told in single-particle cryo-EM; see the references.)

In [ ]:
# ── 2.1 Flatland templates: 2D projections of the zoo (40 × 40 pixels of 10 Å) ──
S2 = 40                                       # flatland image size in pixels
zoo40 = load_zoo(CAST, size=S2, sigma=1.0, verbose=False)
flat = {pid: normalize(zoo40[pid].sum(axis=1)) for pid in CAST}    # projection along y → an image in (z, x)
# (For the 20S proteasome this happens to be an oblique view of the barrel; Section 6 uses true side views.)
fig, axes = plt.subplots(1, 4, figsize=(7.5, 2.1))
for ax, pid in zip(axes, CAST):
    ax.imshow(flat[pid]); ax.set_title(f"{pid.upper()}\n{ZOO[pid]['name']}", fontsize=8); ax.axis("off")
plt.suptitle("2D projections used from here on (40 nm boxes)", fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
# ── 2.2 Align pure noise to a template and average ──
def einstein_from_noise(template, noise, n_angles=36, max_shift=5, batch=100):
    """Exhaustively align each noise image to the template (rotation × shift, max correlation) and return the aligned images."""
    S = template.shape[0]
    angles = 2 * np.pi * np.arange(n_angles) / n_angles
    G = np.conj(np.fft.fft2(np.array([rotate_image(template, a) for a in angles])))
    shifts = np.fft.fftfreq(S) * S
    allowed = (np.abs(shifts)[:, None] <= max_shift) & (np.abs(shifts)[None, :] <= max_shift)
    aligned = []
    for b0 in range(0, len(noise), batch):
        F = np.fft.fft2(noise[b0:b0 + batch])
        cc = np.fft.ifft2(F[:, None] * G[None]).real                # (batch, angles, y, x): every pose at once
        cc = np.where(allowed, cc, -np.inf).reshape(len(F), -1)
        best = cc.argmax(1)
        a, rem = best // (S * S), best % (S * S)
        for i in range(len(F)):
            th, t = angles[a[i]], np.array([shifts[rem[i] % S], shifts[rem[i] // S]])   # noise ≈ rotate(template, th) shifted by t
            c, s = np.cos(th), np.sin(th)
            aligned.append(rotate_image(noise[b0 + i], -th, shift_xy=-(np.array([[c, -s], [s, c]]).T @ t)))
    return np.array(aligned, np.float32)


t0 = time.time()
noise = rng.normal(size=(800, S2, S2)).astype(np.float32)          # 800 images of PURE noise: no molecule anywhere
aligned_ribo = einstein_from_noise(flat["4v4r"], noise)
aligned_prot = einstein_from_noise(flat["3j9i"], noise)            # the SAME noise, a different template
Ns = [10, 30, 100, 300, 800]
curve = [pearson(aligned_ribo[:n].mean(0), flat["4v4r"]) for n in Ns]
print("corr(average of aligned noise, ribosome template):", dict(zip(Ns, np.round(curve, 2))))
print(f"same noise aligned to the proteasome: corr with proteasome {pearson(aligned_prot.mean(0), flat['3j9i']):.2f}, "
      f"with ribosome {pearson(aligned_prot.mean(0), flat['4v4r']):.2f}   ({time.time() - t0:.1f} s)")
assert curve[-1] > 0.4, "the template should clearly emerge from aligned noise"

In [ ]:
# ── 2.3 Figure: something from nothing ──
fig, axes = plt.subplots(1, 5, figsize=(10, 2.4))
panels = [(flat["4v4r"], "template: ribosome"), (aligned_ribo[:30].mean(0), "30 noise images"),
          (aligned_ribo.mean(0), "800 noise images"), (aligned_prot.mean(0), "same 800, aligned\nto the proteasome")]
for ax, (img, title) in zip(axes[:4], panels):
    ax.imshow(img); ax.set_title(title, fontsize=8); ax.axis("off")
axes[4].semilogx(Ns, curve, "o-", color="tab:red")
axes[4].set_xlabel("number of noise images"); axes[4].set_ylabel("corr with template")
axes[4].set_title("bias grows with N", fontsize=8)
plt.tight_layout(); plt.show()

🔍 **What just happened?** There was **no molecule** in any image, yet the average looks like whatever template we used:
a ribosome for the ribosome template, a striped barrel for the proteasome template, from the *same* noise. The
similarity even **grows** with the number of images, exactly like real signal would. Two lessons:

1. A template-based average can only confirm what you put in; it can never surprise you. A complex that is *not* the
   template, or a state the template lacks, will be pulled toward the template.
2. Validation must be independent of the reference: averages of two half-sets aligned to the *same* template agree with
   each other even for pure noise (try it in the ✏️ below), so a high half-set FSC (Module 09) does not prove the
   structure is real if both halves were aligned to one reference. "Gold-standard" refinement aligns the two halves to
   two independently refined references for this reason.

This is why the Xu Lab, and the field, invested in **reference-free** averaging (the rest of this notebook) and
**template-free** discovery (Module 14, Module 20).

✏️ **Your turn.** Compute the Fourier ring correlation between the averages of `aligned_ribo[:400]` and
`aligned_ribo[400:]` with the toolkit's `fsc(...)` and `resolution_at(...)`. What "resolution" does pure noise reach?
<details><summary>Hint</summary><code>f, c, _ = fsc(aligned_ribo[:400].mean(0), aligned_ribo[400:].mean(0), pixel_A=10.0); print(resolution_at(f, c, 0.143))</code></details>

### 📄 Paper connection — Kim, Uddin, Xu & Chang (2023), review

- **Paper**: Kim HH, Uddin MR, Xu M, Chang Y-W. *Computational methods toward unbiased pattern mining and structure
  determination in cryo-electron tomography data.* J Mol Biol 435(9):168068 (2023).
  [doi:10.1016/j.jmb.2023.168068](https://doi.org/10.1016/j.jmb.2023.168068)
- **Question it answers**: which computational tools exist for each step of cryo-ET analysis, and how can the field move
  from template-based, user-guided steps (which bias results and cannot find unknown structures) toward automated,
  *unbiased* pattern discovery?
- **What the paper actually does**: a narrative review organised along the pipeline (segmentation → picking → structure
  determination → classification), contrasting template matching and multireference alignment with unsupervised
  approaches (multi-pattern pursuit, DISCA, Jim-Net, Harmony) and listing open challenges (few annotations, simulation,
  domain adaptation, accessibility, model size, benchmarks).
- **What our toy keeps**: the motivating concern, reference bias of template-based averaging.
- **What our toy leaves out**: everything else; the review contains no experiments. Whether it discusses the specific
  "noise reproduces the template" experiment was not checked, so treat Section 2 as our illustration of its theme.
- **Result reported**: not applicable (review).

---
## 3 · Flatland: a fast 2D stand-in for subtomograms

Classification needs *many* alignments: every particle against every class, every iteration. In 3D each alignment costs
seconds; in a notebook we cannot afford thousands. So from here on we work in **flatland**:

| 3D cryo-ET | our 2D analogue |
|---|---|
| subtomogram $f(x,y,z)$, 32³–64³ voxels | image $f(x,z)$, 40 × 40 pixels (a projection of a zoo map) |
| random 3D rotation $R\in SO(3)$ + shift | random in-plane rotation $\theta$ + shift |
| missing wedge $|\xi_z|\le|\xi_x|\tan60°$ (in Fourier space) | the same formula on the 2D grid: two missing sectors |
| fast rotational search with spherical harmonics (Module 11) | exact rotational search with a 1D FFT along the polar angle (Module 11) |

⚠️ **SNR is not comparable between 2D and 3D.** What matters for alignment is roughly the total signal energy relative
to the noise, which grows with the number of pixels: $J\times\mathrm{SNR}$. A 40² image (1,600 pixels) at SNR 1 holds
about as much as a 32³ subtomogram (32,768 voxels) at SNR 0.05, or a 64³ one at SNR 0.006. So the flatland SNRs below
look generous but correspond to realistic 3D values.

### 3.1 A 2D fast aligner (recap of Module 11)

For two images $f,g$ with Fourier masks $M_f,M_g$, Xu, Beck & Alber (2012) score a rotation $R$ by a **phase-free**,
hence translation-invariant, constrained correlation of Fourier magnitudes:

$$c'(R)=\frac{\int |\mathcal Ff|\,M_f\;\Lambda_R\big[|\mathcal Fg|\,M_g\big]\,d\xi}{\sqrt{\int|\mathcal Ff|^2M_f\,\Lambda_R[M_g]\,d\xi\;\int M_f\,\Lambda_R\big[|\mathcal Fg|^2M_g\big]\,d\xi}}=\frac{c_0(R)}{\sqrt{c_1(R)\,c_2(R)}}.$$

*In words:* compare the magnitude spectra only where **both** images were measured, normalised like a correlation. In 2D
a rotation is a cyclic shift along the polar angle, so each $c_i(\theta)$ for **all** angles at once is one 1D FFT per
ring. The best few angles become candidates; for each, an FFT search over shifts of the full constrained correlation
(with phases) picks the winner. Two practical choices of our toy (not taken from the paper): we use only the Fourier rings
between 200 Å and 33 Å (the lowest rings describe the overall blob and the highest are mostly noise), and we fade the
image edges with a soft circular window so the square box does not dominate the spectrum.

Note that 2D magnitude spectra cannot tell $\theta$ from $\theta+180°$ (a real image's spectrum is point-symmetric); that
is one reason to keep **several** candidate angles and let the phase-sensitive translational search decide.

In [ ]:
# ── 3.1 Flatland geometry: wedges and rigid transforms ──
N_THETA = 180                                  # angular sampling: 2°
ANG = 2 * np.pi * np.arange(N_THETA) / N_THETA
TILT = 60.0
KY, KX = freq_grid((S2, S2))
SHIFTS = np.fft.fftfreq(S2) * S2               # shift (pixels) belonging to each FFT index


def wedge2d(shape, theta=0.0, tilt_max_deg=TILT):
    """2D missing-wedge mask seen from a frame rotated by theta: W(R_θᵀ ξ). theta=0 is the microscope frame."""
    ky, kx = freq_grid(shape)
    c, s = np.cos(theta), np.sin(theta)
    kx2, ky2 = c * kx + s * ky, -s * kx + c * ky
    m = np.abs(ky2) <= np.abs(kx2) * np.tan(np.radians(tilt_max_deg)) + 1e-9
    m[0, 0] = True
    return m.astype(np.float32)


W0 = wedge2d((S2, S2))                                         # every raw particle has this wedge
WEDGES = np.array([wedge2d((S2, S2), a) for a in ANG])         # the wedge after rotating a particle by each grid angle


def transform_image(img, theta, t):
    """T_(θ,t): rotate by θ (counter-clockwise) about the centre, then shift by t = (tx, ty) pixels."""
    return rotate_image(img, theta, shift_xy=t)


def inverse_transform_image(img, theta, t):
    """Undo transform_image(img, theta, t)."""
    c, s = np.cos(theta), np.sin(theta)
    return rotate_image(img, -theta, shift_xy=-(np.array([[c, -s], [s, c]]).T @ np.asarray(t, float)))


def flatland_particles(images, names, snr, rng, max_shift=2.0):
    """Simulate 2D 'subtomograms': random rotation + shift, missing wedge, noise (noise also confined to the wedge)."""
    X, poses = [], []
    for name in names:
        th, t = rng.uniform(0, 2 * np.pi), rng.uniform(-max_shift, max_shift, 2)
        clean = apply_fourier_mask(transform_image(images[name], th, t), W0)
        X.append(apply_fourier_mask(add_noise_snr(clean, snr, rng), W0))
        poses.append((th, t))
    return np.array(X, np.float32), poses


x = flat["4v4r"]
roundtrip = pearson(x, inverse_transform_image(transform_image(x, 0.7, (2, -1)), 0.7, (2, -1)))
print(f"transform → inverse round trip: corr {roundtrip:.3f} (interpolation costs a little)")
assert roundtrip > 0.95

In [ ]:
# ── 3.1 The fast rotational search c'(θ) and the translational search ──
PAD = 2                                          # zero-pad 2× before the FFT: finer sampling of the spectrum
N2 = S2 * PAD
RINGS = np.arange(4, 25).astype(float)           # rings of the padded spectrum: 0.05–0.30 cycles/pixel (200–33 Å)
_yy, _xx = np.indices((S2, S2))
APOD = np.clip((S2 / 2 - 1 - np.sqrt((_yy - (S2 - 1) / 2)**2 + (_xx - (S2 - 1) / 2)**2)) / 3.0, 0, 1)   # soft circular window


def _polar_matrix():
    """Sparse bilinear-interpolation matrix: padded, centred spectrum → (ring, angle) samples."""
    rr, pp = np.meshgrid(RINGS, ANG, indexing="ij")
    y, x = N2 // 2 + rr * np.sin(pp), N2 // 2 + rr * np.cos(pp)
    y0, x0 = np.floor(y).astype(int), np.floor(x).astype(int)
    fy, fx = y - y0, x - x0
    rows = np.repeat(np.arange(y.size), 4)
    cols = np.stack([y0 * N2 + x0, y0 * N2 + x0 + 1, (y0 + 1) * N2 + x0, (y0 + 1) * N2 + x0 + 1], -1).ravel()
    vals = np.stack([(1 - fy) * (1 - fx), (1 - fy) * fx, fy * (1 - fx), fy * fx], -1).ravel()
    return sparse.csr_matrix((vals, (rows, cols)), shape=(y.size, N2 * N2))


POLAR = _polar_matrix()


def polar_spectra(imgs, wedge_angle=0.0):
    """The three ingredients of c' for a batch of images: polar FFTs of |F|·M, |F|²·M and M (M = None → full coverage)."""
    imgs = np.asarray(imgs, float) * APOD
    o = (N2 - S2) // 2
    pad = np.zeros((len(imgs), N2, N2)); pad[:, o:o + S2, o:o + S2] = imgs
    A = np.abs(np.fft.fftshift(np.fft.fft2(pad), axes=(1, 2))); A[:, N2 // 2, N2 // 2] = 0
    m = np.ones((N2, N2)) if wedge_angle is None else np.fft.fftshift(wedge2d((N2, N2), wedge_angle))

    def to_polar(a):
        return (POLAR @ a.reshape(len(a), -1).T).T.reshape(len(a), len(RINGS), N_THETA)
    return (np.fft.fft(to_polar(A * m), axis=-1), np.fft.fft(to_polar(A**2 * m), axis=-1),
            np.fft.fft(to_polar(m[None]), axis=-1)[0])


def rot_scores(a, b):
    """c'(θ) = c0/√(c1·c2) for rotating each image of b onto each image of a, for all N_THETA angles at once."""
    w = RINGS[:, None]                                         # area element r·dr of a ring
    c0 = (np.fft.ifft(a[0] * np.conj(b[0]), axis=-1).real * w).sum(-2)
    c1 = (np.fft.ifft(a[1] * np.conj(b[2]), axis=-1).real * w).sum(-2)
    c2 = (np.fft.ifft(a[2] * np.conj(b[1]), axis=-1).real * w).sum(-2)
    return c0 / np.sqrt(np.maximum(c1 * c2, 1e-12))


def top_peaks(scores, n):
    """Indices of the n highest circular local maxima along the last axis (candidate angles)."""
    is_peak = (scores >= np.roll(scores, 1, -1)) & (scores >= np.roll(scores, -1, -1))
    return np.argsort(-np.where(is_peak, scores, -np.inf), axis=-1)[..., :n]


def translation_search(FX, G, Om, max_shift):
    """Constrained correlation (Xu 2012, Eq. 7 in 2D) over all shifts by one inverse FFT; returns (best score, (tx, ty))."""
    num = np.fft.ifft2(FX * Om * np.conj(G * Om)).real
    den = np.sqrt((np.abs(FX)**2 * Om).sum((-2, -1)) * (np.abs(G)**2 * Om).sum((-2, -1))) / (S2 * S2)
    cc = num / np.maximum(den, 1e-12)[..., None, None]
    allowed = (np.abs(SHIFTS)[:, None] <= max_shift) & (np.abs(SHIFTS)[None, :] <= max_shift)
    cc = np.where(allowed, cc, -np.inf).reshape(cc.shape[:-2] + (-1,))
    k = cc.argmax(-1)
    return np.take_along_axis(cc, k[..., None], -1)[..., 0], np.stack([SHIFTS[k % S2], SHIFTS[k // S2]], -1)


def reference_bank(ref):
    """FFTs of a (full-coverage) reference rotated to every grid angle, mean removed: shape (N_THETA, S2, S2)."""
    F = np.fft.fft2(np.array([rotate_image(ref, a) for a in ANG])); F[:, 0, 0] = 0
    return F


def align_to_reference(FX, PX, ref, n_cand=6, max_shift=3):
    """Fast alignment of every particle (wedge W0) to a full reference: X_i ≈ T_(θ,t)(ref).

    Returns per particle n_cand candidates sorted best first: scores (B, n), angle indices (B, n), shifts (B, n, 2)."""
    bank = reference_bank(ref)
    js = top_peaks(rot_scores(PX, polar_spectra(ref[None], None)), n_cand)
    FXd = FX.copy(); FXd[:, 0, 0] = 0
    best, t = translation_search(FXd[:, None], bank[js], W0, max_shift)
    order = np.argsort(-best, axis=1)
    return (np.take_along_axis(best, order, 1), np.take_along_axis(js, order, 1),
            np.take_along_axis(t, order[..., None], 1))


print("flatland aligner ready:", N_THETA, "angles,", len(RINGS), "Fourier rings")

In [ ]:
# ── 3.2 How accurate is the fast aligner? (ribosome particles vs the clean ribosome) ──
t0 = time.time()
print("SNR   median angle error   fraction within 10°")
for snr in [1.0, 0.3, 0.1, 0.03]:
    Xt, poses_t = flatland_particles(flat, ["4v4r"] * 60, snr, rng)
    _, js_t, _ = align_to_reference(np.fft.fft2(Xt), polar_spectra(Xt, 0.0), flat["4v4r"])
    err = np.abs(np.degrees(np.angle(np.exp(1j * (ANG[js_t[:, 0]] - np.array([p[0] for p in poses_t]))))))
    print(f"{snr:4.2f}   {np.median(err):10.1f}°          {np.mean(err < 10):.2f}")
print(f"({time.time() - t0:.1f} s)")

🔍 **What just happened?** Against a clean reference, the fast aligner finds the in-plane angle to within a few degrees
down to flatland SNR ≈ 0.1 and then fails more and more often. Keep that in mind: when the *best* pose is wrong for a
large fraction of particles, any method that trusts only the best pose is in trouble. Section 5 is about exactly that.

---
## 4 · Alignment through classification (Xu, Beck & Alber 2012)

### 4.1 The idea

Alignment needs class averages (references); class averages need classes; classes need aligned particles. Xu, Beck &
Alber broke this circle by **iterating**, starting without any reference:

1. **Pairwise similarity.** Align particles to each other with fast constrained matching and record the constrained
   correlation $c_{jk}$; turn it into a distance $d_{jk}=\sqrt{2-2c_{jk}}$ (the Euclidean distance between two
   normalised images whose correlation is $c_{jk}$).
2. **Hierarchical clustering** (average linkage), with the tree **cut where the average silhouette width is largest**.
   The silhouette of particle $i$ is
   $$s(i)=\frac{b(i)-a(i)}{\max\{a(i),b(i)\}},$$
   where $a(i)$ is its mean distance to its own cluster and $b(i)$ its mean distance to the nearest other cluster.
   *In words:* $s\approx1$ means "clearly in the right group", $s\approx0$ "on the fence", $s<0$ "probably misplaced"
   (Module 04).
3. **Average** each cluster into a template (wedge-weighted, Section 1) and **align the templates to each other**, so
   that everything lives in one common frame.
4. **Re-align every particle to every template** and **re-assign** it to the best one.
5. **Feature enhancement.** Up-weight the voxels where aligned particles vary most consistently:
   $$W(x)=\frac{1}{|N_x|}\sum_{y\in N_x}\mathrm{COV}_{xy},\qquad \mathrm{COV}_{xy}=\frac1K\sum_{k=1}^K\big(f^k_x-\bar f_x\big)\big(f^k_y-\bar f_y\big),$$
   the average covariance between voxel $x$ and its neighbours $N_x$ over the $K$ aligned particles. *In words:* where
   neighbouring voxels rise and fall *together* across particles there is structure that differs between particles,
   i.e. information about the classes; isolated noise does not co-vary with its neighbours.
6. Compute weighted constrained correlations in the common frame, re-cluster, and **repeat until the silhouette stops
   improving**.

Our toy follows these steps in flatland. One choice is ours: at the very first round we cut the tree into **more**
groups than we expect (6). Merging two groups of the same molecule later is easy (their averages align onto each
other), whereas a group that mixes two molecules gives a blurred average that keeps attracting both.

In [ ]:
# ── 4.2 The data: a ribosome / proteasome mixture in flatland ──
CLASSES_4 = ["4v4r", "3j9i"]                     # two of our stars
labels_4 = np.repeat(np.arange(2), 25)           # 25 ribosomes + 25 proteasomes
X4, poses_4 = flatland_particles(flat, [CLASSES_4[l] for l in labels_4], snr=0.3, rng=np.random.default_rng(1))
fig, axes = plt.subplots(1, 8, figsize=(10, 1.6))
for ax, i in zip(axes, [0, 1, 2, 3, 25, 26, 27, 28]):
    ax.imshow(X4[i]); ax.set_title(ZOO[CLASSES_4[labels_4[i]]]["name"].split()[-1], fontsize=7); ax.axis("off")
plt.suptitle("Flatland particles at SNR 0.3 (random rotation, shift and missing wedge). Can you sort them?", fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
# ── 4.3 Step 1: all-pairs fast constrained alignment ──
BAND = (np.sqrt(KX**2 + KY**2) <= 0.25).astype(np.float32)     # compare only frequencies up to 1/(40 Å)


def pairwise_alignment(X, n_cand=2, max_shift=3):
    """Align every pair with the fast aligner (both particles carry wedges).

    Returns C (N×N constrained correlations) and poses TH, TT such that X[i] ≈ T_(TH[i,j], TT[i,j])(X[j]) for i < j."""
    N = len(X)
    FX = np.fft.fft2(X); FX[:, 0, 0] = 0
    P0, P1, PM = polar_spectra(X, 0.0)
    C, TH, TT = np.eye(N), np.zeros((N, N)), np.zeros((N, N, 2))
    for i in range(N - 1):
        js = np.arange(i + 1, N)
        cand = top_peaks(rot_scores((P0[i], P1[i], PM), (P0[js], P1[js], PM)), n_cand)
        G = np.fft.fft2(np.array([[rotate_image(X[j], ANG[a]) for a in cand[q]] for q, j in enumerate(js)]))
        G[..., 0, 0] = 0
        Om = W0 * WEDGES[cand] * BAND                           # coefficients measured in BOTH particles
        best, t = translation_search(FX[i][None, None], G, Om, max_shift)
        k = best.argmax(1)
        C[i, js] = C[js, i] = best[np.arange(len(js)), k]
        TH[i, js] = ANG[cand[np.arange(len(js)), k]]
        TT[i, js] = t[np.arange(len(js)), k]
    return C, TH, TT


t0 = time.time()
C4, TH4, TT4 = pairwise_alignment(X4)
print(f"{len(X4) * (len(X4) - 1) // 2} pairwise alignments in {time.time() - t0:.1f} s")
within = C4[np.ix_(labels_4 == 0, labels_4 == 0)][~np.eye(25, dtype=bool)].mean()
between = C4[np.ix_(labels_4 == 0, labels_4 == 1)].mean()
print(f"mean pairwise correlation: ribosome–ribosome {within:.3f}, ribosome–proteasome {between:.3f}")

In [ ]:
# ── 4.4 Step 2: hierarchical clustering and the silhouette ──
def silhouette_cut(C, k_range=range(2, 9)):
    """Average-linkage tree on d = √(2−2c); return (best silhouette, K, labels, linkage, all silhouettes) over k_range."""
    D = np.sqrt(np.clip(2 - 2 * C, 0, None)); np.fill_diagonal(D, 0)
    Z = linkage(squareform(D, checks=False), method="average")
    best, scores = None, {}
    for k in k_range:
        lab = fcluster(Z, k, criterion="maxclust") - 1
        if len(np.unique(lab)) < 2:
            continue
        scores[k] = silhouette_score(D, lab, metric="precomputed")
        if best is None or scores[k] > best[0]:
            best = (scores[k], len(np.unique(lab)), lab)
    return best[0], best[1], best[2], Z, scores


sil0, K0, lab0, Z0, sil_scores = silhouette_cut(C4)
_, _, lab_init, _, _ = silhouette_cut(C4, k_range=[6])          # our deliberate over-partition for round 0
print("silhouette per K:", {k: round(v, 3) for k, v in sil_scores.items()})
print(f"best silhouette cut: K = {K0}, ARI vs truth = {adjusted_rand_score(labels_4, lab0):.2f};  "
      f"over-partition K = 6: ARI = {adjusted_rand_score(labels_4, lab_init):.2f}")
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
dendrogram(Z0, ax=axes[0], no_labels=True, color_threshold=0)
axes[0].set_title("average-linkage tree of round 0"); axes[0].set_ylabel("distance √(2−2c)")
axes[1].plot(list(sil_scores), list(sil_scores.values()), "o-")
axes[1].set_xlabel("number of clusters K"); axes[1].set_ylabel("average silhouette width")
axes[1].set_title("where to cut the tree?")
plt.tight_layout(); plt.show()

🔍 **What just happened?** With the particles still in their random poses, pairwise alignment gives a noisy similarity
matrix: within-class pairs correlate only slightly better than between-class pairs, and the first tree is far from
perfect (look at the ARI, the adjusted Rand index of Module 04: 1 = perfect, 0 = chance). One round is not enough; the
power of the method is in the **iteration**.

In [ ]:
# ── 4.5 Steps 3–6: tools for the common frame ──
def compose_to_frame(phi, psi):
    """Particle x ≈ T_phi(template); template_0 ≈ T_psi(template)  →  (α, u) with T_(α,u)(x) in template_0's frame."""
    (th, t), (thp, s) = phi, psi
    a = thp - th
    c, sn = np.cos(a), np.sin(a)
    return a, np.asarray(s, float) - np.array([[c, -sn], [sn, c]]) @ np.asarray(t, float)


def to_common(X, alphas, us):
    """Move particles into the common frame; return the images and their (rotated) wedge masks."""
    Xc = np.array([transform_image(x, a, u) for x, a, u in zip(X, alphas, us)])
    return Xc, np.array([wedge2d((S2, S2), a) for a in alphas])


def wedge_average(Xc, Mc):
    """Missing-wedge-weighted average (Section 1) of particles already in a common frame."""
    F = (np.fft.fft2(Xc) * Mc).sum(0)
    n = Mc.sum(0)
    return np.real(np.fft.ifft2(np.where(n > 0, F / np.maximum(n, 1), 0)))


def feature_weights(Xc):
    """Xu 2012 feature enhancement: mean covariance of each pixel with its 8 neighbours across particles (≥ 0, max 1)."""
    D = Xc - Xc.mean(0)
    cov = np.mean([(D * np.roll(D, (dy, dx), axis=(1, 2))).mean(0)
                   for dy in (-1, 0, 1) for dx in (-1, 0, 1) if (dy, dx) != (0, 0)], axis=0)
    W = np.clip(cov, 0, None)
    return W / (W.max() + 1e-12)


def weighted_cc_matrix(Xc, Mc, W):
    """All-pairs weighted constrained correlation in the common frame (only coefficients measured by both), via matrix products."""
    G = np.fft.fft2(Xc * W[None]); G[:, 0, 0] = 0
    M = (Mc * BAND[None]).reshape(len(Xc), -1)
    Gm = G.reshape(len(Xc), -1) * M
    P = np.abs(G.reshape(len(Xc), -1))**2 * M
    return np.real(Gm @ np.conj(Gm).T) / np.maximum(np.sqrt((P @ M.T) * (M @ P.T)), 1e-12)


def align_full(ref, img, n_cand=4, max_shift=4):
    """Align a full-coverage image (e.g. a class average) onto another: ref ≈ T_(θ,t)(img). Returns (score, θ, t)."""
    js = top_peaks(rot_scores(polar_spectra(ref[None], None), polar_spectra(img[None], None))[0], n_cand)
    G = np.fft.fft2(np.array([rotate_image(img, ANG[j]) for j in js])); G[:, 0, 0] = 0
    F = np.fft.fft2(ref); F[0, 0] = 0
    best, t = translation_search(F[None], G, np.ones((S2, S2)), max_shift)
    k = best.argmax()
    return best[k], ANG[js[k]], t[k]


print("common-frame tools ready")

In [ ]:
# ── 4.6 The alignment-through-classification loop ──
def alignment_through_classification(X, C, TH, TT, n_iter=5, k_first=6, labels_true=None):
    """Xu, Beck & Alber (2012)-style loop in flatland. Returns final labels, templates and a per-round history."""
    N = len(X)
    _, K, lab, _, _ = silhouette_cut(C, k_range=[k_first])
    templates = []
    for k in range(K):                                          # round-0 templates: align members to the cluster medoid
        mem = np.where(lab == k)[0]
        m = mem[np.argmax(C[np.ix_(mem, mem)].mean(1))]
        poses = [(0.0, np.zeros(2)) if j == m else (TH[m, j], TT[m, j]) if m < j
                 else compose_to_frame((TH[j, m], TT[j, m]), (0.0, np.zeros(2))) for j in mem]
        Xc, Mc = to_common(X[mem], *zip(*poses))
        templates.append(wedge_average(Xc, Mc))
    history = [dict(round=0, K=K, silhouette=np.nan, ari=adjusted_rand_score(labels_true, lab) if labels_true is not None else np.nan)]
    FX, PX = np.fft.fft2(X), polar_spectra(X, 0.0)
    for it in range(1, n_iter + 1):
        K = len(templates)
        psi = [(0.0, np.zeros(2))] + [align_full(templates[0], templates[k])[1:] for k in range(1, K)]   # step 3
        scores, best_pose = np.zeros((N, K)), []
        for k in range(K):                                                                     # step 4
            s_, j_, t_ = align_to_reference(FX, PX, templates[k], n_cand=4, max_shift=3)
            scores[:, k] = s_[:, 0]
            best_pose.append((ANG[j_[:, 0]], t_[:, 0]))
        assign = scores.argmax(1)
        frame = [compose_to_frame((best_pose[a][0][i], best_pose[a][1][i]), psi[a]) for i, a in enumerate(assign)]
        Xc, Mc = to_common(X, *zip(*frame))
        W = feature_weights(Xc)                                                                # step 5
        sil, K, lab, _, _ = silhouette_cut(weighted_cc_matrix(Xc, Mc, W))                      # step 6
        templates = [wedge_average(Xc[lab == k], Mc[lab == k]) for k in range(K)]
        history.append(dict(round=it, K=K, silhouette=sil,
                            ari=adjusted_rand_score(labels_true, lab) if labels_true is not None else np.nan))
    return lab, templates, history, W


t0 = time.time()
lab_atc, templates_atc, hist_atc, W_atc = alignment_through_classification(X4, C4, TH4, TT4, n_iter=5,
                                                                           labels_true=labels_4)
for h in hist_atc:
    print(f"round {h['round']}: K = {h['K']}, silhouette = {h['silhouette']:.3f}, ARI vs truth = {h['ari']:.2f}")
print(f"({time.time() - t0:.1f} s)")
assert hist_atc[-1]["ari"] > hist_atc[0]["ari"], "iterating should improve on the first rough clustering"

In [ ]:
# ── 4.7 Figure: the loop converges ──
fig = plt.figure(figsize=(10, 2.8))
ax = fig.add_subplot(1, 4, 1)
r = [h["round"] for h in hist_atc]
ax.plot(r, [h["ari"] for h in hist_atc], "o-", label="ARI vs truth")
ax.plot(r[1:], [h["silhouette"] for h in hist_atc[1:]], "s--", label="silhouette")
ax.set_xlabel("round"); ax.set_ylim(-0.05, 1.05); ax.legend(fontsize=7); ax.set_title("progress", fontsize=9)
ax = fig.add_subplot(1, 4, 2)
ax.imshow(W_atc, cmap="viridis"); ax.set_title("feature weights W(x)", fontsize=9); ax.axis("off")
for j, tmpl in enumerate(templates_atc[:2]):
    ax = fig.add_subplot(1, 4, 3 + j)
    ax.imshow(tmpl); ax.axis("off")
    best = max(CLASSES_4, key=lambda p: align_full(tmpl, flat[p])[0])
    ax.set_title(f"class {j} average\n(looks like {ZOO[best]['name']})", fontsize=8)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Starting from a deliberately over-split, half-wrong first clustering, each round
(re)aligned every particle to every class average, moved all particles into one frame, re-weighted pixels by their
neighbour covariance, and re-clustered. The two groups of the same molecule merged, the silhouette rose, and the
classes came to match the truth (ARI → 1) without anyone providing a template. The feature-weight map lights up where
ribosome and proteasome differ, the "informative" pixels.

⚠️ **Limitations you should know.**
- The silhouette favours the *most obvious* split. If two of three classes are similar (try the ✏️ below), the cut
  often stops at K = 2. The paper had the same criterion; real pipelines therefore inspect classes and sub-classify.
- Every particle is assigned to **one** best class and **one** best pose. When the best pose is wrong (low SNR), the
  error is baked in. Section 5 relaxes exactly that.

✏️ **Your turn.** (a) In 4.2 change the data seed `np.random.default_rng(1)` to `np.random.default_rng(4)` and re-run
4.2–4.7. With some seeds the first round is so poor that the loop never recovers: iterative methods inherit the quality
of their start. (b) Set `CLASSES_4 = ["4v4r", "3j9i", "1bxn"]` and `labels_4 = np.repeat(np.arange(3), 20)` in 4.2 and
re-run. Does the silhouette find three classes? Which two get merged, and do their projections (Section 2.1) explain why?

### 📄 Paper connection — Xu, Beck & Alber (2012), the classification part

- **Paper**: Xu M, Beck M, Alber F. *High-throughput subtomogram alignment and classification by Fourier space
  constrained fast volumetric matching.* J Struct Biol 178(2):152–164 (2012).
  [doi:10.1016/j.jsb.2012.02.014](https://doi.org/10.1016/j.jsb.2012.02.014) (open access: PMC3821800)
- **Question it answers**: how to align and classify thousands of subtomograms *without a reference*, fast enough to be
  practical, despite the missing wedge.
- **What the paper actually does**: fast rotational search over all rotations at once (spherical harmonics + FFT,
  Module 11) on a phase-free constrained correlation, then FFT translational search; for classification, the iterative
  scheme of Section 4.1: feature-enhancement weights $W(x)$ from neighbour covariances, pairwise weighted constrained
  correlations turned into distances $\sqrt{2-2c}$, average-linkage clustering cut at the maximal average silhouette,
  class averaging, template-to-template alignment, re-alignment and re-assignment, until the silhouette plateaus.
- **What our toy keeps**: the loop structure, constrained (both-wedges) correlation, the $\sqrt{2-2c}$ distance, the
  silhouette cut, wedge-weighted class averages, and neighbour-covariance feature weights.
- **What our toy leaves out**: 3D (we use flatland), spherical-harmonic rotational search, the paper's exact weighted
  correlation (its Eq. 13), large datasets; and it adds our own over-partitioned first round and a frequency band.
- **Result reported**: the authors report 100 % classification of four simulated complexes after 18 iterations with
  feature enhancement; 68.2 % on 786 experimental GroEL / GroEL–GroES subtomograms (clustering failed without feature
  enhancement); 99 % for ribosome vs ATP synthase from *Leptospira interrogans* tomograms; and alignment about 6,000×
  faster than exhaustive scanning (153 s vs about 10 days for 64³ volumes at 2.5° sampling).
- **Code**: AITom `aitom/align/fast/util.py` (alignment; C++ core `aitom/tomominer/core/src/align.cpp`).

---
## 5 · Maximum likelihood with fast alignment (FAML, Zhao *et al.* 2018)

### 5.1 A generative model of a subtomogram

Instead of *choosing* one class and one pose per particle, write down how a particle is **generated**:

$$X_i = R_{\phi_i}A_{\kappa_i} + G_i,\qquad G_i\sim\mathcal N(0,\sigma^2),$$

where $A_1,\dots,A_K$ are the class averages (unknown), $\kappa_i$ is particle $i$'s class (hidden), $\phi_i$ its pose,
rotation and translation (hidden), and $G_i$ Gaussian noise. Only the Fourier coefficients inside the wedge, $X_i^o$
(mask $w_i$), are observed. The class is drawn with prior probability $\alpha_k$ and the shift
$q=(q_x,q_y,q_z)$ from a Gaussian of width $\zeta$. **Maximum likelihood (ML)** chooses $A,\alpha,\sigma,\zeta$ to make
the observed data most probable, *averaging over* every possible class and pose of every particle:

$$\mathcal L(\Theta)=\sum_i\log\sum_{k}\int_{\phi}P(X_i^o\mid k,\phi,\Theta)\,P(k,\phi\mid\Theta)\,d\phi .$$

*In words:* a good model makes every particle likely under *some* class and pose; we do not need to know which.

### 5.2 EM: alternate "who is where" and "what do they look like"

The EM algorithm (Module 04) alternates two steps.

**E-step (soft assignment).** For each particle, the posterior probability of every (class, pose):

$$e(\phi,k)=\alpha_k\exp\Big\{-\frac{\sum_j\big\|(R_\phi A_k)_j-X^o_{ij}\big\|^2}{2\sigma^2}-\frac{q_x^2+q_y^2+q_z^2}{2\zeta^2}\Big\},\qquad
P(k,\phi\mid X_i^o)=\frac{e(\phi,k)}{\sum_{k'}\sum_{\phi'}e(\phi',k')\,\tilde\upsilon(\phi')}.$$

*In words:* a (class, pose) is probable if the rotated class average matches the observed coefficients well (small
squared difference, measured in units of the noise variance), if the class is common, and if the shift is small.

**M-step (weighted averaging).** Every particle contributes to every class average in every pose, weighted by its
posterior; coefficients the particle did not measure are filled with the current average (they are latent too):

$$A^{new}_{kj}=\frac{1}{N\alpha^{new}_k}\sum_i\sum_{\phi}P(k,\phi\mid X^o_i)\,\tilde\upsilon(\phi)\Big\{[R_\phi^{-1}w_i]_j\,(R_\phi^{-1}X^o_i)_j+\big[1-(R_\phi^{-1}w_i)_j\big]A_{kj}\Big\},$$

$$\alpha_k^{new}=\frac1N\sum_i\sum_\phi P\,\tilde\upsilon,\qquad
\sigma^{2,new}=\frac1{NJ}\sum_{i,j,k,\phi}P\,\tilde\upsilon\,\big\{w_{ij}\|(R_\phi A_k)_j-X^o_{ij}\|^2+(1-w_{ij})\sigma^2\big\},\qquad
\zeta^{2,new}=\frac{1}{3N}\sum P\,\tilde\upsilon\,(q_x^2+q_y^2+q_z^2).$$

($J$ is the number of Fourier coefficients; in flatland the shift has 2 components, so we divide by $2N$.)

### 5.3 The FAML trick: integrate over a *few* good poses

The integral over all rotations and translations is what makes ML expensive (RELION-style methods sample it finely).
FAML replaces it by a sum over a **small set ⊕ of candidate poses**, the local maxima of the fast rotational score
$c'$ (Section 3), each refined by the FFT translational search, typically fewer than 50 per particle and class:

$$\int_\phi f(\phi)\,d\phi\;\approx\;\sum_{\phi\in\oplus}f(\phi)\,\tilde\upsilon(\phi,\oplus).$$

The weight $\tilde\upsilon$ is the relative **volume of the candidate's Voronoi cell** in pose space (the set of poses
closer to this candidate than to any other), estimated by Monte Carlo with the distance
$\tau(\phi_1,\phi_2)=\sqrt{\|I-R_1^{\mathsf T}R_2\|_F^2+\|\phi^{tr}_1-\phi^{tr}_2\|^2}$. *In words:* a candidate that
stands alone in a large empty region of pose space "represents" more poses than one of a tight bunch, so it gets more
weight. The **hard** alternative ("fast alignment", FA) keeps only the single best-scoring (class, pose) per particle.

In [ ]:
# ── 5.4 Monte-Carlo Voronoi weights in flatland pose space (θ, tx, ty) ──
def voronoi_weights(thetas, shifts, rng, max_shift=3, n_samples=2000):
    """Relative Voronoi-cell volume of each candidate pose, by Monte Carlo.

    thetas: (..., n) candidate angles; shifts: (..., n, 2). Distance τ² = ‖I − R1ᵀR2‖²_F + ‖t1 − t2‖², and for 2D
    rotations ‖I − R1ᵀR2‖²_F = 4(1 − cos(θ1 − θ2))."""
    th_s = rng.uniform(0, 2 * np.pi, n_samples)
    t_s = rng.uniform(-max_shift, max_shift, (n_samples, 2))
    d2 = (4 * (1 - np.cos(thetas[..., :, None] - th_s))
          + ((shifts[..., :, None, :] - t_s)**2).sum(-1))              # (..., n, n_samples)
    nearest = d2.argmin(-2)                                              # which candidate owns each sample
    n = thetas.shape[-1]
    counts = np.stack([(nearest == c).sum(-1) for c in range(n)], -1)
    return counts / n_samples


demo_w = voronoi_weights(np.array([0.0, 0.05, 0.10, np.pi]), np.zeros((4, 2)), np.random.default_rng(0))
print("three bunched candidates near 0° and one alone at 180°  →  weights", np.round(demo_w, 2))

⚠️ **Code-reading note.** In the AITom version we read (`aitom/geometry/volume/hypervolume/util.py`,
`voronoi_weights_6d`, commit `dc5b275`), the nearest-candidate search sits *after* the Monte-Carlo sampling loop instead
of inside it, so only the last random sample is counted (each weight vector becomes one-hot); the three Euler angles are
also drawn uniformly, which is not uniform over rotations (Module 02). FAML's EM can also run with equal weights
(`use_voronoi=False`). We implement the estimate described in the paper. Reading the code next to the paper is a
research skill: the two do not always agree.

In [ ]:
# ── 5.5 FAML-style EM (soft) and its hard-assignment (FA) counterpart, in flatland ──
def faml_em(X, A0, n_iter=8, soft=True, n_cand=6, max_shift=3, use_voronoi=True, seed=0, verbose=False):
    """EM over classes and a few candidate poses per (particle, class) — the FAML approximation, 2D analogue.

    soft=True : posterior-weighted averaging (ML).  soft=False: every particle goes, whole, to its single best
    (class, pose) by constrained correlation (FA). Returns averages, labels, and a per-iteration history."""
    rng_v = np.random.default_rng(seed)
    N, K = len(X), len(A0)
    FX, PX = np.fft.fft2(X), polar_spectra(X, 0.0)
    A = np.array(A0, float)
    alpha = np.ones(K) / K
    sigma2 = float((np.abs(FX)**2 * W0).sum() / (N * W0.sum()))        # start: total observed power per coefficient
    zeta2 = 4.0
    history = []
    for it in range(n_iter):
        sc, js, ts, r2 = (np.zeros((N, K, n_cand)), np.zeros((N, K, n_cand), int),
                          np.zeros((N, K, n_cand, 2)), np.zeros((N, K, n_cand)))
        for k in range(K):                                     # candidate poses from the fast aligner (the ⊕ set)
            sc[:, k], js[:, k], ts[:, k] = align_to_reference(FX, PX, A[k], n_cand, max_shift)
            FA_rot = np.fft.fft2(np.array([rotate_image(A[k], a) for a in ANG]))
            shift_phase = np.exp(-2j * np.pi * (KX * ts[:, k, :, 0, None, None] + KY * ts[:, k, :, 1, None, None]))
            r2[:, k] = (np.abs(FA_rot[js[:, k]] * shift_phase - FX[:, None])**2 * W0).sum((-2, -1))
        log_e = np.log(alpha)[None, :, None] - r2 / (2 * sigma2) - (ts**2).sum(-1) / (2 * zeta2)
        log_v = np.log(voronoi_weights(ANG[js], ts, rng_v, max_shift) + 1e-12) if use_voronoi else 0.0
        if soft:                                               # E-step: posterior over (class, candidate pose)
            P = np.exp(log_e + log_v - logsumexp((log_e + log_v).reshape(N, -1), axis=1)[:, None, None])
        else:                                                  # FA: all weight on the best constrained correlation
            P = np.zeros_like(log_e)
            P.reshape(N, -1)[np.arange(N), sc.reshape(N, -1).argmax(1)] = 1.0
        # M-step. Back-rotating is linear, so we first sum the (back-shifted) particles that share an angle.
        FA_old, A_new = np.fft.fft2(A), np.zeros_like(A)
        for k in range(K):
            back = FX[:, None] * np.exp(2j * np.pi * (KX * ts[:, k, :, 0, None, None] + KY * ts[:, k, :, 1, None, None]))
            acc = np.zeros((N_THETA, S2, S2), complex); wsum = np.zeros(N_THETA)
            np.add.at(acc, js[:, k].ravel(), (P[:, k, :, None, None] * back).reshape(-1, S2, S2))
            np.add.at(wsum, js[:, k].ravel(), P[:, k].ravel())
            num = np.zeros((S2, S2), complex)
            for j in np.where(wsum > 1e-8)[0]:
                xb = rotate_image(np.real(np.fft.ifft2(acc[j])), -ANG[j])
                Wr = WEDGES[(-j) % N_THETA]                      # what these particles measured, in the class frame
                num += Wr * np.fft.fft2(xb) + wsum[j] * (1 - Wr) * FA_old[k]   # unmeasured → current average
            A_new[k] = np.real(np.fft.ifft2(num / max(P[:, k].sum(), 1e-9)))
        sigma2 = float(((P * r2).sum() + N * (1 - W0).sum() * sigma2) / (N * S2 * S2))
        zeta2 = float(max((P * (ts**2).sum(-1)).sum() / (2 * N), 0.25))
        alpha = P.sum((0, 2)) / N + 1e-6; alpha /= alpha.sum()
        A = A_new
        n_eff = 1.0 / (P.reshape(N, -1)**2).sum(1)             # "effective number" of (class, pose) per particle
        history.append(dict(labels=P.sum(2).argmax(1), alpha=alpha.copy(), A=A.copy(), n_eff=float(n_eff.mean())))
        if verbose:
            print(f"  iter {it}: alpha = {np.round(alpha, 2)}, sigma² = {sigma2:.3g}, ζ² = {zeta2:.2f}, "
                  f"effective candidates per particle = {n_eff.mean():.2f}")
    return A, history[-1]["labels"], history


def corr_to_truth(A, truth):
    """Correlation of an average with the true image after aligning the truth onto it (rotation + shift)."""
    s_, th, t = align_full(A, truth, max_shift=6)
    return pearson(A, transform_image(truth, th, t))


print("FAML-style EM ready")

In [ ]:
# ── 5.6 One run, up close: K = 2 classes, starting from the Section 4 averages (FA first, then ML) ──
t0 = time.time()
labels_5 = np.repeat(np.arange(2), 30)
X5, _ = flatland_particles(flat, [CLASSES_4[l] for l in labels_5], snr=0.1, rng=np.random.default_rng(5))
A_init = np.array(templates_atc[:2])                      # class averages found without any template in Section 4
A5, lab5, hist5 = faml_em(X5, A_init, n_iter=6, soft=True, n_cand=10, verbose=True)
print(f"classification accuracy (Hungarian-matched) per iteration:",
      [round(hungarian_accuracy(labels_5, h["labels"]), 2) for h in hist5], f"  ({time.time() - t0:.1f} s)")

🔍 **What just happened?** Each iteration the E-step spread every particle over its 2 classes × 10 candidate poses;
the printed "effective candidates per particle" ($1/\sum P^2$, which equals 1 when one hypothesis takes all the
weight) shows how undecided the particles are: about two at first, close to one at the end, as the averages and hence
the posteriors sharpen. Because we started from the good averages of Section 4, the classification is right from the
first iteration; ML refines poses and averages rather than rescuing a bad start. Note that ML needs no separate
"assign" step: a particle's class is simply the one with the largest posterior mass.

### 5.7 Soft (ML) versus hard (FA) as the SNR drops

The cleanest test of the FAML claim is **reference-free averaging** of one class from a featureless start (the
rotational average of all particles, a round blob), at decreasing SNR. Both methods use exactly the same candidate
poses (20 per particle) and the same M-step; the only difference is whether each particle is spread over its
candidates by posterior weight (soft) or given entirely to its best-scoring candidate (hard).

In [ ]:
# ── 5.7 Soft (ML) vs hard (FA) as the SNR drops (K = 1, reference-free start) ──
def rotational_average(img):
    """Average an image over all in-plane rotations (a featureless, unbiased starting reference)."""
    y, x = np.indices(img.shape)
    r = np.sqrt((y - (img.shape[0] - 1) / 2)**2 + (x - (img.shape[1] - 1) / 2)**2).astype(int)
    profile = np.bincount(r.ravel(), img.ravel()) / np.maximum(np.bincount(r.ravel()), 1)
    return profile[r]


t0 = time.time()
sweep = {}
for snr in [0.1, 0.03, 0.01]:
    Xs, _ = flatland_particles(flat, ["4v4r"] * 64, snr, np.random.default_rng(57))
    start = rotational_average(Xs.mean(0))[None]
    for soft in [True, False]:
        A_s, _, h_s = faml_em(Xs, start, n_iter=4, soft=soft, n_cand=20, seed=1)
        sweep[(snr, soft)] = (corr_to_truth(A_s[0], flat["4v4r"]), h_s[-1]["n_eff"], A_s[0])
    print(f"SNR {snr:5.2f}: corr with true ribosome  soft (ML) {sweep[(snr, True)][0]:.3f}  vs  hard (FA) "
          f"{sweep[(snr, False)][0]:.3f}   | effective candidates per particle (soft) {sweep[(snr, True)][1]:.1f}")
print(f"({time.time() - t0:.1f} s)")

In [ ]:
# ── 5.7 Figure: soft vs hard ──
snrs = [0.1, 0.03, 0.01]
fig = plt.figure(figsize=(10, 3))
ax = fig.add_subplot(1, 4, 1)
ax.semilogx(snrs, [sweep[(s, True)][0] for s in snrs], "o-", label="soft (ML, FAML-style)")
ax.semilogx(snrs, [sweep[(s, False)][0] for s in snrs], "s--", label="hard (best pose, FA)")
ax.invert_xaxis(); ax.set_xlabel("flatland SNR (→ harder)"); ax.set_ylabel("corr with true ribosome")
ax.legend(fontsize=7); ax.set_title("reference-free averaging", fontsize=9)
for j, (soft, name) in enumerate([(True, "soft"), (False, "hard")]):
    ax = fig.add_subplot(1, 4, 2 + j)
    ax.imshow(sweep[(0.01, soft)][2]); ax.axis("off"); ax.set_title(f"{name}, SNR 0.01", fontsize=9)
ax = fig.add_subplot(1, 4, 4)
ax.imshow(flat["4v4r"]); ax.axis("off"); ax.set_title("truth", fontsize=9)
plt.tight_layout(); plt.show()

🔍 **What just happened?** At the highest SNR the two methods are close: the best pose is almost always right, so
spreading weight over other candidates changes little. As the SNR drops, the best-scoring pose is more and more often wrong
(Section 3.2), and the hard average is built partly from misaligned copies, while the soft average keeps a share of
every particle on the candidates that are still plausible; the printed "effective candidates per particle" grows as the
data get noisier. In our runs the soft (ML) average stays closer to the truth at the two lower SNRs. This is the
qualitative claim of FAML (ML is more robust at low SNR), reproduced in miniature; the size of the gap in a toy like
this depends on the seed, the number of candidates and the number of iterations (✏️ below), so do not read too much into
the exact numbers. The paper's 3D experiments, with many more voxels per particle, report a much larger gap.

✏️ **Your turn.** (a) In 5.7 change `n_cand=20` to `n_cand=3`. With so few candidates, can soft assignment still help?
(b) Use `n_iter=8` instead of `n_iter=4`: both averages may get *worse* with more iterations at the lowest
SNR. Why can iterating an EM on very noisy data over-fit the noise (think of Section 2)?

### 📄 Paper connection — Zhao, Zeng, Guo & Xu (2018), FAML

- **Paper**: Zhao Y, Zeng X, Guo Q, Xu M. *An integration of fast alignment and maximum-likelihood methods for electron
  subtomogram averaging and classification.* Bioinformatics 34(13):i227–i236 (ISMB 2018).
  [doi:10.1093/bioinformatics/bty267](https://doi.org/10.1093/bioinformatics/bty267) ·
  [arXiv:1804.01203](https://arxiv.org/abs/1804.01203)
- **Question it answers**: ML averaging/classification is robust but must integrate over all rotations and translations
  (expensive); fast alignment is cheap but commits to one transform (fragile at low SNR or with large wedges). Can we get
  the robustness at close to the cost?
- **What the paper actually does**: the generative model and EM of Sections 5.1–5.2 in Fourier space, with missing
  coefficients treated as latent and imputed from the current average; the pose integral replaced by a sum over fewer
  than 50 sub-optimal transforms from fast alignment (local maxima of the rotational upper bound, each refined by FFT
  translation search), weighted by Monte-Carlo Voronoi volumes under the metric $\tau$.
- **What our toy keeps**: the model, the posterior, the four M-step updates (including imputation), the candidate set
  from fast rotational search, Voronoi weights, and the soft-vs-hard comparison.
- **What our toy leaves out**: 3D, 6D pose space, the paper's datasets and initialisation, parallelism, and any
  regularisation; our 2D SNRs are not comparable to 3D ones (Section 3).
- **Result reported**: the authors report correlation with the true structure of 0.77 for FAML vs 0.19 for FA on GroEL
  at SNR 0.003; 0.78 vs 0.49 with a limited tilt range and a preferred orientation; on experimental mixtures a mean F1 of
  0.863 vs 0.462 (TRiC / capped proteasome) and correlations 0.87/0.78 vs 0.40/0.24 (GroEL / GroEL–GroES); and runtime
  for 20 subtomograms at 64³ on 12 processes of FA 106 s, FAML 451 s, RELION 2041 s.
- **Code**: AITom `aitom/average/ml/faml/faml.py`; tutorial `doc/tutorials/009_faml.py`.

---
## 6 · From classes to biology: how many proteasomes carry a cap?

The singly capped proteasome (6UTJ) is the 20S barrel (3J9I) plus a regulatory particle on one end, a **compositional**
state. **Conformational** states (the same parts, rearranged) raise the same computational problem. Thalassinos *et al.*
(2013) argue that such states are best characterised by **integrating** several kinds of data: cryo-EM/ET for shape and
location, mass spectrometry for composition and stoichiometry, atomic structures for detail. Classification turns a
pile of subtomograms into a **number with an uncertainty**, the occupancy of each state, and a **difference map** that
says where the states differ. Those are what biologists compare across conditions (e.g. with and without a drug).

We classify a flatland mixture of 35 bare and 25 capped barrels with the FAML-style EM. Here we know which two states to
look for, so we start from the two known structures **blurred to 60 Å**, a standard multi-reference start. (The blur
limits reference bias, but Section 2 says you must still check that the result does not merely echo the references:
see the ✏️ below.) The two barrels look alike in projection, and in our tests the fully reference-free loop of Section 4
did not separate them reliably, which is itself a realistic lesson: subtle states are the hardest to discover.

In [ ]:
# ── 6.1 Classify a 20S / capped-20S mixture ──
t0 = time.time()
STATES = ["3j9i", "6utj"]
# Compare like with like: SIDE views of both barrels (the 7-fold axis in the image plane). In the PolNet maps the 3J9I
# barrel axis lies in the xy plane and the 6UTJ axis along z, so we project 3J9I along z and 6UTJ along y.
side = {"3j9i": normalize(zoo40["3j9i"].sum(axis=0)), "6utj": normalize(zoo40["6utj"].sum(axis=1))}
labels_6 = np.repeat(np.arange(2), [35, 25])                 # truth: 35 bare, 25 capped (hidden from the method)
X6, _ = flatland_particles(side, [STATES[l] for l in labels_6], snr=0.3, rng=np.random.default_rng(6))
init = np.array([lowpass(side[p], 1 / 6.0) for p in STATES])  # known structures, blurred to 60 Å (cycles/pixel = 10/60)
A6, lab6, hist6 = faml_em(X6, init, n_iter=6, soft=True, n_cand=10)
print(f"accuracy per iteration: {[round(hungarian_accuracy(labels_6, h['labels']), 2) for h in hist6]}"
      f"   ({time.time() - t0:.1f} s)")

In [ ]:
# ── 6.2 Occupancy with a bootstrap confidence interval, and the difference map ──
# Which class is the capped one? The capped barrel is longer: compare the extent of each average above half its maximum.
extent = [np.ptp(np.argwhere(A > 0.5 * A.max()), axis=0).max() for A in A6]
capped = int(np.argmax(extent))
is_capped = (lab6 == capped).astype(float)
boot = np.array([is_capped[np.random.default_rng(b).integers(0, len(is_capped), len(is_capped))].mean()
                 for b in range(2000)])
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"capped fraction: {is_capped.mean():.2f}  (95 % bootstrap CI {lo:.2f}–{hi:.2f});  truth: {labels_6.mean():.2f}")
_, th_d, t_d = align_full(A6[capped], A6[1 - capped])                  # put the bare barrel onto the capped one
bare_on_capped = transform_image(A6[1 - capped], th_d, t_d)
diff = normalize(A6[capped]) - normalize(bare_on_capped)
fig, axes = plt.subplots(1, 4, figsize=(9.5, 2.6))
axes[0].imshow(A6[1 - capped]); axes[0].set_title("class: bare 20S", fontsize=9)
axes[1].imshow(A6[capped]); axes[1].set_title("class: capped", fontsize=9)
im = axes[2].imshow(diff, cmap="RdBu_r", vmin=-np.abs(diff).max(), vmax=np.abs(diff).max())
axes[2].set_title("difference (capped − bare)", fontsize=9); plt.colorbar(im, ax=axes[2], fraction=0.046)
for ax in axes[:3]:
    ax.axis("off")
axes[3].hist(boot, bins=30, color="tab:purple"); axes[3].axvline(labels_6.mean(), color="k", ls="--", label="truth")
axes[3].set_xlabel("capped fraction"); axes[3].set_title("bootstrap", fontsize=9); axes[3].legend(fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** The classification turned 60 noisy images into two biological quantities: the **fraction of
capped proteasomes** with a 95 % bootstrap confidence interval (resampling particles with replacement, Module 04), and a
**difference map** whose strongest positive density sits at one end of the barrel: the cap. With 60 particles the
interval is wide; quadrupling the number of particles roughly halves it. These are the numbers you would compare
between, say, untreated and drug-treated cells (Module 25), always together with their uncertainty.

✏️ **Your turn.** Replace the starting references in 6.1 by two identical featureless blobs,
`init = np.array([rotational_average(X6.mean(0))] * 2) + 0.01 * np.random.default_rng(0).normal(size=(2, S2, S2))`,
and re-run 6.1–6.2. Does the EM still find the cap? What does this tell you about how much the result above depended on
knowing the two structures in advance?

### 📄 Paper connection — Thalassinos, Pandurangan, Xu, Alber & Topf (2013), perspective

- **Paper**: Thalassinos K, Pandurangan AP, Xu M, Alber F, Topf M. *Conformational states of macromolecular assemblies
  explored by integrative structure calculation.* Structure 21(9):1500–1508 (2013).
  [doi:10.1016/j.str.2013.08.006](https://doi.org/10.1016/j.str.2013.08.006)
- ⚠️ We could not access the full text of this paper while writing; the description below is limited to the title and
  abstract. Read the paper before relying on details.
- **What it is**: a **review / perspective**, not a methods paper. It surveys computational methods for two data types
  used to study assemblies that switch between conformations: **mass spectrometry**, and **3D cryo-EM with a focus on
  aligning and classifying heterogeneous subtomograms from cryo-ET** (the Xu-lab contribution area, Sections 4–5), and
  discusses integrating them with atomic structures to describe assembly dynamics in the native environment.
- **What our toy keeps**: the idea that classification yields state populations and difference maps that can be combined
  with other data.
- **What our toy leaves out**: mass spectrometry and any integrative modelling (see Module 25).
- **Result reported**: not applicable (perspective).

---
## ✅ Checkpoint

1. 400 aligned subtomograms at SNR 0.02 are averaged. What SNR do you expect in the average, and what assumption makes
   this work?
   <details><summary>Answer</summary>About 400 × 0.02 = 8, assuming the noise is independent between copies and the copies are perfectly aligned and identical.</details>
2. Why do we divide each Fourier coefficient by the number of copies that measured it, rather than by $N$?
   <details><summary>Answer</summary>Because each copy lacks a (differently rotated) wedge; dividing by N under-weights coefficients that few copies measured and distorts the average direction-dependently. Dividing by the count n(ξ) gives an unbiased average of the copies that actually saw ξ.</details>
3. You align 10,000 noise-only patches from an empty region of a tomogram to a ribosome template and get a
   ribosome-like average. What does this prove?
   <details><summary>Answer</summary>Nothing about the sample: it shows reference bias ("Einstein from noise"). The average reflects the template plus selection, not real particles.</details>
4. In the alignment-through-classification loop, what is the role of aligning the class averages to each other?
   <details><summary>Answer</summary>It puts all particles into one common frame, so their pairwise correlations are meaningful and two clusters that are the same molecule in different poses can be merged.</details>
5. In FAML, what is approximated by the candidate set ⊕, and why does that save time?
   <details><summary>Answer</summary>The integral of the likelihood over all rotations and translations. Summing over fewer than ~50 candidate poses from fast alignment (instead of a fine grid of millions of poses) makes each E-step cheap.</details>
6. When is soft (ML) assignment expected to help most compared with hard (best-pose) assignment?
   <details><summary>Answer</summary>When the best-scoring pose/class is often wrong but the correct one still has non-negligible likelihood, i.e. at low SNR, with large missing wedges or preferred orientations; then averaging over several hypotheses avoids committing to the wrong one.</details>

## 🧠 Questions & discussion

1. Section 1 showed that particles with a preferred orientation leave a missing cone. Which kinds of biological samples
   are prone to this, and how could you collect data differently to avoid it?
2. "Einstein from noise" is extreme. Describe a realistic situation in cellular cryo-ET where *partial* reference bias
   could mislead you (for example a ribosome template used on a mixture of ribosomes and similarly sized complexes).
3. The silhouette criterion chose a sensible number of classes in our two-class example but not always with three
   classes. Propose another way to choose the number of classes and say what it would need (hint: Module 04's BIC,
   Module 14's SFSC).
4. Soft assignment spreads each particle over several poses. Can this blur the average? When would you prefer hard
   assignment?
5. FAML's candidate poses come from fast alignment. What happens if the correct pose is not among the candidates? How
   would you detect that in practice?
6. If caspofungin-treated cells (Module 25) showed 40 % instead of 25 % capped proteasomes, what extra controls would you
   need before claiming a biological effect?

## 🚀 Bonus — going further

- **Real code.** Read AITom's `aitom/average/ml/faml/faml.py` and its tutorial `doc/tutorials/009_faml.py`; map each
  function (`ln_p_xo`, `update_a`, `update_sigma`, …) to an equation in Section 5.2.
- **3D.** Replace flatland by 24³ subtomograms with a coarse 3D rotation grid (Module 11) for the FAML E-step; how many
  candidates per particle do you need?
- **Gold standard.** Repeat Section 2 with two half-sets aligned to two *different* references and compare the FRC with
  the single-reference case.
- **More classes.** Run Section 4 with four classes and 30 particles each; record when the silhouette merges classes.
- **Deep dives** in this repository: [`../../SubtomogramAveraging/`](../../SubtomogramAveraging/) and, for continuous
  heterogeneity (cryoDRGN-style), [`../../HeterogeneousReconstruction/`](../../HeterogeneousReconstruction/).

## 📝 Lab note

```text
Module 13 — date:
Input: PolNet maps (4V4R, 3J9I, 6UTJ, 1BXN); 3D subtomograms (Section 1) and flatland particles (Sections 2–6)
Output: wedge-weighted averages, an Einstein-from-noise average, classes and averages from two classification methods
Method: Fourier-count weighting; exhaustive alignment of noise; Xu 2012 loop (pairwise CC, silhouette cut, feature
        weights); FAML-style EM (candidate poses, Voronoi weights, soft vs hard)
Parameter I changed and what happened:
A failure mode / surprise:
Paper link: Xu, Beck & Alber 2012; Zhao et al. 2018; Kim et al. 2023; Thalassinos et al. 2013
```

## 📚 References

1. Xu M, Beck M, Alber F. High-throughput subtomogram alignment and classification by Fourier space constrained fast volumetric matching. *J Struct Biol* 178:152–164 (2012). doi:10.1016/j.jsb.2012.02.014
2. Zhao Y, Zeng X, Guo Q, Xu M. An integration of fast alignment and maximum-likelihood methods for electron subtomogram averaging and classification. *Bioinformatics* 34:i227–i236 (2018). doi:10.1093/bioinformatics/bty267
3. Kim HH, Uddin MR, Xu M, Chang YW. Computational methods toward unbiased pattern mining and structure determination in cryo-electron tomography data. *J Mol Biol* 435:168068 (2023). doi:10.1016/j.jmb.2023.168068
4. Thalassinos K, Pandurangan AP, Xu M, Alber F, Topf M. Conformational states of macromolecular assemblies explored by integrative structure calculation. *Structure* 21:1500–1508 (2013). doi:10.1016/j.str.2013.08.006
5. Rousseeuw PJ. Silhouettes: a graphical aid to the interpretation and validation of cluster analysis. *J Comput Appl Math* 20:53–65 (1987). doi:10.1016/0377-0427(87)90125-7
6. Henderson R. Avoiding the pitfalls of single particle cryo-electron microscopy: Einstein from noise. *PNAS* 110:18037–18041 (2013). doi:10.1073/pnas.1314449110
7. Scheres SHW, Chen S. Prevention of overfitting in cryo-EM structure determination. *Nat Methods* 9:853–854 (2012). doi:10.1038/nmeth.2115
8. Martinez-Sanchez A *et al.* PolNet (source of the density templates). <https://github.com/anmartinezs/polnet>

## ➡️ Next

**[14 · Discovering the unknown: template-free detection and de novo structural pattern mining](14_template_free_discovery.ipynb)**:
what if you do not know which molecules are in the cell at all? Course overview: [`README.md`](../README.md).